# 08 · Gold assumptions and comments

For the "Ask HawkerReach" Genie space (#126). Runs after `05_gold_closures`:
- writes `assumptions.json` (the file the app and notebooks read) to `gold.model_assumptions`, with `built_at`, the data date the chat shows
- re-applies `Sql/02_gold_comments.sql`, because the gold notebooks overwrite their tables and drop comments

In [0]:
import datetime, json, os
from pyspark.sql import functions as F
from dq import DQLog  # dq.py at the repo root

CATALOG= "hawkerreach"
dbutils.widgets.text("target_schema", "gold")  # dev_<name> for test runs
OUT= f"{CATALOG}.{dbutils.widgets.get('target_schema')}"
print("writing to", OUT)
dq= DQLog(spark, "08_gold_assumptions", schema=f"{CATALOG}.silver" if OUT.endswith(".gold") else OUT)

In [0]:
# one row per assumptions.json key; value is text because one value is a list ("10, 20")
if not os.path.exists("assumptions.json"):
    raise FileNotFoundError(f"assumptions.json not found in {os.getcwd()}: run this notebook from a Git folder clone of the repo")
with open("assumptions.json") as f:
    A= json.load(f)

built_at= datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)  # the job's last gold write
rows= [(k, v["name"], str(v["value"]), v["unit"], v["tag"], v["status"], v["source"], built_at) for k, v in A.items()]
schema= "key string, name string, value string, unit string, tag string, status string, source string, built_at timestamp"
spark.createDataFrame(rows, schema=schema).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OUT}.model_assumptions")

m= spark.table(f"{OUT}.model_assumptions")
dq.step("model_assumptions: keys", len(A), m.count(), f"built_at {built_at:%Y-%m-%d %H:%M} UTC")
assert m.count() == len(A), "a key is missing from model_assumptions"
assert m.select("key").distinct().count() == m.count(), "duplicate key in model_assumptions"

In [0]:
# comments for the seven Genie tables; only on the real gold run, since the file names hawkerreach.gold
if OUT == f"{CATALOG}.gold":
    with open(os.path.join("Sql", "02_gold_comments.sql")) as f:
        sql= "\n".join(l for l in f.read().split("\n") if not l.strip().startswith("--"))
    statements= [s.strip() for s in sql.split(";\n") if s.strip()]
    for s in statements:
        spark.sql(s.rstrip(";"))
    print(f"applied {len(statements)} comment statements")

    tables= ["centres", "centre_closures", "closure_scenarios", "access_sensitivity", "centre_demand", "subzone_seniors", "model_assumptions"]
    missing= [(f"{t}.{c.name}", "no comment") for t in tables for c in spark.table(f"{OUT}.{t}").schema if not c.metadata.get("comment")]
    dq.issues("gold_column_comments", missing)
    assert not missing, f"columns without a comment: {missing}"
else:
    print("skipped comments: test run")